# W02 · Positional encoding & the Lissajous view
# W02 · 位置編碼與 Lissajous 視角

**English.** Absolute positional encoding (APE) maps `x` to
`[x, sin(2^i pi x), cos(2^i pi x), ...]` for `i=1,...,L`. PEPS
reinterprets each frequency as a point moving on a **Lissajous curve**:
`S_i=(1+sin(x phi_i))/2`, `C_i=(1+cos(x phi_i))/2`. This notebook reproduces
paper Fig. 2 and shows the Identity-encoder affine equivalence to APE.

**繁體中文.** 絕對位置編碼(APE)把 `x` 映成
`[x, sin(2^i pi x), cos(2^i pi x), ...]`(`i=1,...,L`)。PEPS 把每個頻率
重新詮釋為在 **Lissajous 曲線**上移動的點:`S_i=(1+sin)/2`、
`C_i=(1+cos)/2`。本 notebook 重現論文 Fig.2,並展示 Identity encoder
與 APE 的仿射等價。

In [1]:
# Repo bootstrap: make `peps` and `apps` importable from the notebook.
import sys, os
sys.path.insert(0, os.path.abspath('..'))
import torch
from peps.train import auto_device
device = auto_device()
print('torch', torch.__version__, '| device', device)

torch 2.10.0+rocm7.0 | device cuda


/opt/amdgpu/share/libdrm/amdgpu.ids: No such file or directory
/opt/amdgpu/share/libdrm/amdgpu.ids: No such file or directory
/opt/amdgpu/share/libdrm/amdgpu.ids: No such file or directory
/opt/amdgpu/share/libdrm/amdgpu.ids: No such file or directory


## 1. Lissajous point motion (reproduce Fig. 2) / Lissajous 點運動(重現 Fig.2)

In [2]:
import torch, matplotlib.pyplot as plt
from peps import Projector
L = 4
proj = Projector(num_frequencies=L, include_input=True)
xs = torch.linspace(0, 1, 400).unsqueeze(1).repeat(1, 2)  # 2D coord sweep
pts = proj(xs)   # (400, 2L+1, 2)
print('num points per coord =', proj.num_points)

fig, ax = plt.subplots(figsize=(5, 5))
for p in range(pts.shape[1]):
    ax.plot(pts[:, p, 0], pts[:, p, 1], lw=1)
ax.set_title('Lissajous trajectories of projected points')
ax.set_xlabel('x-channel'); ax.set_ylabel('y-channel'); ax.set_aspect('equal')
plt.show()

num points per coord = 9


## 2. Identity PEPS is affinely equivalent to APE / Identity PEPS 與 APE 仿射等價
If the shared encoder is the identity, PEPS's projected+concatenated features
are an affine transform of APE features, so they carry the same information.
若共享編碼器為 identity,PEPS 的投影+串接特徵是 APE 特徵的仿射變換,
因此兩者仿射等價。

In [3]:
from peps import IdentityEncoder, AbsolutePositionalEncoding
x = torch.rand(500, 2)
pts = proj(x).reshape(x.shape[0], -1)                 # PEPS+Identity features
ape = AbsolutePositionalEncoding(2, L, include_input=True)(x)
print('PEPS feat dim', pts.shape[1], '| APE feat dim', ape.shape[1])
# least-squares fit APE -> PEPS: near-zero residual proves affine equivalence
A = torch.cat([ape, torch.ones(x.shape[0], 1)], 1)
sol = torch.linalg.lstsq(A, pts).solution
resid = (A @ sol - pts).abs().max().item()
print(f'max residual = {resid:.2e}  -> affine-equivalent' )

PEPS feat dim 18 | APE feat dim 18
max residual = 3.58e-07  -> affine-equivalent


## 3. Takeaway / 小結
PEPS *generalizes* APE: swap the identity encoder for a learned grid and the
same projection machinery becomes a powerful learned encoder (W03-W05).

PEPS 是 APE 的**泛化**:把 identity 換成可學習 grid,同一套投影機制就變成強大的
可學習編碼器(W03-W05)。